
##Bronze Layer


####Step-1 Define Schema as String

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType



orders_schema= StructType({
    StructField("order_id",StringType(),True),
    StructField("order_date",StringType(),True),
    StructField("customer_id",StringType(),True),
    StructField("product_id",StringType(),True),
    StructField("product_name",StringType(),True),
    StructField("category",StringType(),True),
    StructField("quantity",StringType(),True),
    StructField("unit_price",StringType(),True),
    StructField("discount",StringType(),True),
    StructField("total_amount",StringType(),True),
    StructField("payment_method",StringType(),True),
    StructField("store_location",StringType(),True),
    StructField("order_status",StringType(),True),

})

customer_schema= StructType({
    StructField("customer_id", StringType(),True),
    StructField("customer_name",StringType(),True),
    StructField("email",StringType(),True),
    StructField("phone",StringType(),True),
    StructField("gender",StringType(),True),
    StructField("date_of_birth",StringType(),True),
    StructField("city",StringType(),True),
    StructField("state",StringType(),True),
    StructField("regestration_date",StringType(),True),
    StructField("loyality_points",StringType(),True),
    StructField("status",StringType(),True),
})


###Step-2 Read Raw as csv

In [0]:

## here header were npt mapping with the row data 
order_bronze_df= spark.read.format("csv")\
                            .option("header","true")\
                            .schema(orders_schema)\
                            .option("inferSchema","true")\
                            .load("/Volumes/retail_project/raw_data/raw_file/retail_orders_messy.csv")
        
                            

order_bronze_df.show()

In [0]:
order_bronze_df= order_bronze_df.toDF("order_id","order_date","customer_id","product_id",
             "product_name","category","quantity","unit_price",
             "discount","total_amount","payment_method",
             "store_location","order_status")
                              



order_bronze_df.show()



In [0]:
cust_bronze_df= spark.read.format("csv")\
                   .option("header","true")\
                    .option("inferSchema","true")\
                   .schema(customer_schema)\
                   .load("/Volumes/retail_project/raw_data/raw_file/retail_customers_messy.csv")


cust_bronze_df= cust_bronze_df.toDF("customer_id","customer_name","email","phone","gender","date_of_birth","city","state","regestration_date","loyality_points","status")

cust_bronze_df.display()


In [0]:
cust_df.printSchema()


###Step 3: Add ingestion metadata

In [0]:
from pyspark.sql import functions as f
from pyspark.sql.types import *

order_bronze_df= order_bronze_df.withColumn("ingest_timestamp",f.current_timestamp())
order_bronze_df.display()


cust_bronze_df= cust_bronze_df.withColumn("ingested_time", f.current_timestamp())
cust_bronze_df.display()


###Step: 4 Save Data as Delta Table in bronze

In [0]:
order_bronze_df.write.format("delta").mode("overwrite").saveAsTable('retail_project.bronze.orders_bronze')

cust_bronze_df.write.format("delta").mode("overwrite").saveAsTable('retail_project.bronze.customer_bronze')